### Split Source Data into Monthly Batches

#### Purpose

This notebook splits the Germany energy dataset into separate monthly CSV files. Instead of having one large file covering multiple years, this creates smaller, organized files grouped by year and month. This makes the data easier to process incrementally and prepares it for batch-based ingestion into the Bronze layer.

#### Why split into monthly batches?

* **Easier incremental processing** - Process one month at a time instead of the entire dataset
* **Better organization** - Files are stored in a clear folder structure: `year=2015/month=01/`
* **Supports batch tracking** - Each monthly file can be tracked as a separate batch in the lakehouse
* **Reduces memory pressure** - Smaller files are easier to handle during transformations

#### Output structure

Each monthly file is saved with a clean name pattern:
```
/Volumes/.../raw/energy/year=2015/month=01/germany_energy_2015_01.csv
/Volumes/.../raw/energy/year=2015/month=02/germany_energy_2015_02.csv
/Volumes/.../raw/energy/year=2015/month=03/germany_energy_2015_03.csv
...
```

#### Step 1: Load project configuration

This imports the shared configuration from the setup notebook, which provides:
* `landing_volume_path` - Where the source files are stored
* `germany_selected_file` - Path to the Germany-only CSV created in the previous notebook
* `germany_source_columns` - List of columns to keep

In [0]:
%run ../00_setup/03-project-config  

#### Step 2: Read the source and identify available batches

This step:

1. **Reads the Germany-only CSV** from the landing volume
2. **Adds timestamp columns** - Converts the `utc_timestamp` string into an actual timestamp, then extracts the year and month
3. **Finds all unique year-month combinations** - Identifies which months are available in the dataset (e.g., 2015-01, 2015-02, etc.)
4. **Displays the batch list** - Shows you all the monthly batches that will be created

The output table shows every year-month combination that exists in the source data. Each row represents one monthly file that will be generated in the next step.

In [0]:
from pyspark.sql.functions import col, to_timestamp, year, month, date_format

batch_root_path = f"{landing_volume_path}/raw/energy"

germany_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(germany_selected_file)
)
# creating columns to show the timestamp, and based on that we have year and month by passing the function month and year 
germany_df = (
    germany_df
    .withColumn("timestamp_utc", to_timestamp(col("utc_timestamp")))
    .withColumn("batch_year", year(col("timestamp_utc")))
    .withColumn("batch_month", month(col("timestamp_utc")))
)

available_batches = (
    germany_df
    .select("batch_year", "batch_month")
    .dropDuplicates()
    .orderBy("batch_year", "batch_month")
)

display(available_batches)

#### Step 3: Write monthly batch files

##### How this works:

**For each year-month combination** (e.g., 2015-01, 2015-02, etc.):

1. **Filter** - Keep only rows for that specific month
2. **Create path** - Build folder structure like `year=2015/month=01/`
3. **Write CSV** - Save the filtered data to a temporary location
4. **Rename** - Move the Spark-generated `part-*.csv` file to a clean name like `germany_energy_2015_01.csv`
5. **Cleanup** - Delete the temporary folder

##### Result:

Instead of one large file, you get organized monthly files:
* `germany_energy_2015_01.csv`
* `germany_energy_2015_02.csv`
* `germany_energy_2015_03.csv`
* ... and so on


In [0]:
batch_rows = available_batches.collect()

for row in batch_rows:
    batch_year = row["batch_year"]
    batch_month = row["batch_month"]

    monthly_df = (
        germany_df
        .filter((col("batch_year") == batch_year) & (col("batch_month") == batch_month))
        .drop("timestamp_utc", "batch_year", "batch_month")
    )

    month_text = str(batch_month).zfill(2)

    temp_output_path = f"{batch_root_path}/year={batch_year}/month={month_text}/temp"
    final_output_path = f"{batch_root_path}/year={batch_year}/month={month_text}/germany_energy_{batch_year}_{month_text}.csv"

    (
        monthly_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(temp_output_path)
    )
   #  Look inside temp folder
#     Find the file that starts with part-
#     Make sure it ends with .csv
#     Take the first matching file 
# final_output_path is the name we WANT
# part_file is the file Spark ACTUALLY created

    part_file = [
        file.path
        for file in dbutils.fs.ls(temp_output_path)
        if file.name.startswith("part-") and file.name.endswith(".csv")
    ][0]

    dbutils.fs.rm(final_output_path, recurse=True)
    dbutils.fs.mv(part_file, final_output_path) # move from part file to final output
    dbutils.fs.rm(temp_output_path, recurse=True)

    print(f"created monthly batch: {final_output_path}")

#### Verification

The output above shows each monthly file that was created. You can now:

* Navigate to the landing volume to see the organized folder structure
* Verify that each year/month folder contains exactly one CSV file
* Confirm the file naming follows the pattern `germany_energy_YYYY_MM.csv`

These monthly batch files are now ready to be ingested into the Bronze layer one month at a time.

In [0]:
display(dbutils.fs.ls(f"{landing_volume_path}/raw/energy/year=2019/month=01"))